# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [5]:
"""
%%bash

# Make sure uv is in PATH for this session
export PATH="$HOME/.local/bin:$PATH"

# Install uv
wget -qO- https://astral.sh/uv/install.sh | sh

# Ensure PATH again (installer doesn’t affect current shell)
export PATH="$HOME/.local/bin:$PATH"

# Remove/recreate venv cleanly
uv venv .venv --seed --clear

# Verify uv works
uv --version

# Install dependencies
.venv/bin/python -m pip install \
    sympy \
    numpy \
    transformers \
    vllm \
    tqdm \
    bitsandbytes \
    antlr4-python3-runtime==4.11.1 \
    ipykernel \
    jupyter

# Install Jupyter kernel
.venv/bin/python -m ipykernel install \
    --user \
    --name cse151b \
    --display-name "Python (cse151b)"
"""


downloading uv 0.11.11 x86_64-unknown-linux-gnu


installing to /home/tsinha/.local/bin


  uv


  uvx


everything's installed!


...

 typer-0.25.1 typing-extensions-4.15.0 typing-inspection-0.4.2 tzdata-2026.2 uri-template-1.3.0 urll

ib3-2.6.3 uvicorn-0.46.0 uvloop-0.22.1 vllm-0.20.1 watchfiles-1.1.1 wcwidth-0.7.0 webcolors-25.10.0 

webencodings-0.5.1 websocket-client-1.9.0 websockets-16.0 widgetsnbextension-4.0.15 xgrammar-0.2.0 y

arl-1.23.0 z3-solver-4.15.4.0 zipp-3.23.1


Installed kernelspec cse151b in /home/tsinha/.local/share/jupyter/kernels/cse151b


### Run the cell below every time to activate the installed environment. 

In [1]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate



In [2]:
!nvidia-smi

Sun May 10 04:41:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.20             Driver Version: 580.126.20     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A30                     On  |   00000000:C4:00.0 Off |                    0 |
| N/A   35C    P0             35W /  165W |       0MiB /  24576MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [3]:
import json
import os
import re
import sys
from pathlib import Path
from typing import Optional

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 32768


## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [4]:

data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [5]:
SYSTEM_PROMPT_MATH = """
You are solving a math problem for an automatic grader. The grader strips everything inside <think>...</think> and extracts your answer from \\boxed{} in the post-think text. It checks symbolic equivalence using sympy with 1e-8 relative tolerance.

# How to think
Reason step by step inside your thinking block. Verify each step (arithmetic, signs, domains, units, what variable was actually asked). Length there does not affect your score — but a truncated trace with no \\boxed{} does. If you sense you're running long, commit a best-guess answer immediately and stop.

You may write multiple \\boxed{} expressions inside <think>...</think> while exploring — they are stripped before grading. Only the post-think block matters.

# Final answer block (this is everything that gets scored)
After your thinking ends, write a brief 1-3 line summary, then this exact closing line:

Therefore, the final answer is \\boxed{...}.

# What goes inside \\boxed{}
- VALUES ONLY. No "x =", no labels, no prose.
- EXACT symbolic forms. Decimals lose: write \\frac{1}{2} not 0.5, \\sqrt{2} not 1.414, \\frac{\\pi}{4} not 0.785, \\ln(2) not 0.693, e^2 not 7.389, \\arctan(4.76) not atan(4.76).
- Decimals only when the problem explicitly says so ("to 3 decimal places", "round to...").
- Multi-part: comma-separate IN THE ORDER ASKED inside one box.

# Format examples by answer type

Single value:
Therefore, the final answer is \\boxed{\\frac{\\sqrt{3}}{2}}.

Multiple values (in the order asked):
Therefore, the final answer is \\boxed{3, \\frac{1}{2}, \\sqrt{5}}.

Set (unordered, e.g. solutions to x^2 = 9):
Therefore, the final answer is \\boxed{\\{-3, 3\\}}.

Interval (use \\cup for unions):
Therefore, the final answer is \\boxed{(-\\infty, 2) \\cup (2, \\infty)}.

Equation of a line / curve:
Therefore, the final answer is \\boxed{y = 2x + 1}.

True/False:
Therefore, the final answer is \\boxed{True}.

Very small or very large number:
Therefore, the final answer is \\boxed{2.5 \\times 10^{-7}}.

± answers:
Therefore, the final answer is \\boxed{\\pm 3}    (or \\boxed{\\{-3, 3\\}})

# Pre-box checklist (run silently before writing the box)
1. Did I answer the variable/quantity actually asked, not an intermediate?
2. Right sign? Right count of values? Right order for multi-part?
3. Exact symbolic form, not a decimal (unless asked for one)?

# Hard rules for the post-think block
- Exactly one \\boxed{} expression. (If giving multiple values, put them in ONE box, comma-separated.)
- Inside the box: values only — no "x =", no \\text{...} wrappers, no units.
- If you run out of budget, commit your best current answer in \\boxed{} immediately. A boxed guess beats an unboxed correct answer.
""".strip()


SYSTEM_PROMPT_MCQ = """
You are answering a multiple-choice math problem for an automatic grader. One of the listed options is correct — your job is to identify the letter, not to re-derive the answer from scratch with full rigor.

# Strategy
- Solve enough to distinguish the correct option from the distractors. You usually do not need a complete derivation.
- When it's faster, plug candidate options back into the problem rather than solving forward.
- If your computed result matches one option (numerically or symbolically), pick that option — do not redo the work.
- Eliminate options aggressively: ruling out 3 of 4 is as good as solving.

# How to think
Reason step by step inside your thinking block — content there is stripped before grading, so length is fine. If you're running long or unsure, stop and commit your best guess. A guessed letter scores better than no letter.

# Final answer block
After thinking, write a brief 1-2 line justification, then close with exactly:

Therefore, the answer is \\boxed{X}.

where X is a single uppercase letter only — no parentheses, no period, no option text.

Correct: \\boxed{C}
Wrong:   \\boxed{(C)}, \\boxed{C.}, \\boxed{C) 7}, \\boxed{Option C}

Before boxing, verify: does the letter you chose actually correspond to the option you believe is correct? (Common mistake: solving correctly but boxing the wrong letter.)
""".strip()



def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $2a$
H. $frac{3}{2a}$
I. $frac{3}{2a^2}$
J. ...

── Free-form user prompt (first 200 chars) ──
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS] ...



## 5. Load Model with vLLM

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [6]:
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.50,
    max_model_len=16384,
    trust_remote_code=True,
    max_num_seqs=256,
    max_num_batched_tokens=32768,
 )

sampling_params = SamplingParams(
    #n=5,                  # self-consistency: 5 samples per prompt
    max_tokens=MAX_TOKENS,
    temperature=0.2,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
 )

print("Model loaded.")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

(EngineCore pid=483) 

INFO 05-10 04:47:56 [core.py:299] init engine (profile, create kv cache, warmup model) took 88.63 s (compilation: 60.47 s)


(EngineCore pid=483) 

INFO 05-10 04:47:57 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])


Model loaded.


### 6.2 QLoRA Training

Load the base model in **4-bit NF4** via BitsAndBytes, attach LoRA adapters to all attention and MLP projection layers, then fine-tune with `SFTTrainer`.

| Component | VRAM |
|---|---|
| 4-bit weights | ~2.5 GB |
| LoRA params + optimizer | ~0.5 GB |
| Activations (batch=1, seq=4096) | ~2.5 GB |
| **Total** | **~5.5 GB** ✓ |

In [13]:
#%pip install peft trl accelerate datasets -q

Note: you may need to restart the kernel to use updated packages.


In [7]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# NOTE: This QLoRA section is optional. To use it, you must first prepare train_dataset
# from correct response pairs (see Section 6.1 comments for details).
# For now, this cell is a template showing how to set up QLoRA fine-tuning.

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

qlora_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
qlora_model = prepare_model_for_kbit_training(qlora_model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
qlora_model = get_peft_model(qlora_model, lora_config)
qlora_model.print_trainable_parameters()

# Uncomment below to train (requires train_dataset to be prepared first)
# trainer = SFTTrainer(
#     model=qlora_model,
#     tokenizer=tokenizer,
#     train_dataset=train_dataset,
#     args=SFTConfig(
#         output_dir="./qlora_checkpoints",
#         per_device_train_batch_size=1,
#         gradient_accumulation_steps=8,
#         num_train_epochs=3,
#         learning_rate=2e-4,
#         bf16=True,
#         max_seq_length=4096,
#         dataset_text_field="text",
#         logging_steps=10,
#         save_strategy="epoch",
#         warmup_ratio=0.05,
#         lr_scheduler_type="cosine",
#         report_to="none",
#     ),
# )
# trainer.train()
# print("QLoRA training complete.")

print("QLoRA setup complete. Uncomment trainer.train() above to begin fine-tuning.")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

/home/tsinha/private/151B_SP26_Competition/.venv/lib/python3.13/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145
QLoRA setup complete. Uncomment trainer.train() above to begin fine-tuning.


In [9]:
ADAPTER_PATH = "./qlora_adapter"
qlora_model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"LoRA adapter saved to {ADAPTER_PATH}")

LoRA adapter saved to ./qlora_adapter


In [12]:
### 6.3 Merge & Reload for Inference

Merge the LoRA weights into the base model on CPU (avoids 8 GB VRAM pressure during the merge), save the checkpoint, then load it via vLLM. The final cell rebinds `llm = llm_ft` so the rest of the notebook (Sections 7–10) transparently runs against the fine-tuned model.

SyntaxError: invalid character '–' (U+2013) (2492351549.py, line 3)

In [14]:
from peft import PeftModel

# Free everything that may still be on GPU before loading the merged model:
# the SFTTrainer holds the model + optimizer + grads (~10 GB), and any
# leftover vLLM engine from earlier sections is also still resident.
for _name in ("trainer", "qlora_model", "llm", "llm_data", "llm_ft"):
    if _name in globals():
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f"GPU free after cleanup: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")

MERGED_PATH = "./qlora_merged"

print("Merging adapter into base model (CPU)...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="cpu",
    trust_remote_code=True,
)
peft_model   = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
merged_model = peft_model.merge_and_unload()
merged_model.save_pretrained(MERGED_PATH)
tokenizer.save_pretrained(MERGED_PATH)
del base_model, peft_model, merged_model
gc.collect()
torch.cuda.empty_cache()
print(f"Merged model saved to {MERGED_PATH}")

llm_ft = LLM(
    model=MERGED_PATH,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    gpu_memory_utilization=0.50,
    max_model_len=8192,
    trust_remote_code=True,
    max_num_seqs=2,
    enforce_eager=True,
)

# Re-bind `llm` so Section 7 (Generate Responses) and downstream cells
# automatically use the fine-tuned model.
llm = llm_ft
print("Fine-tuned model ready for inference (bound to `llm`).")



GPU free after cleanup: 25.10 GB
Merging adapter into base model (CPU)...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved to ./qlora_merged
INFO 05-09 23:44:19 [utils.py:233] non-default args: {'trust_remote_code': True, 'load_format': 'bitsandbytes', 'max_model_len': 8192, 'gpu_memory_utilization': 0.5, 'max_num_seqs': 2, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'enforce_eager': True, 'model': './qlora_merged'}


INFO 05-09 23:44:19 [model.py:555] Resolved architecture: Qwen3ForCausalLM


INFO 05-09 23:44:19 [model.py:1680] Using max model len 8192


WARNING 05-09 23:44:19 [vllm.py:896] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none


WARNING 05-09 23:44:19 [vllm.py:914] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.


INFO 05-09 23:44:19 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'])


INFO 05-09 23:44:19 [vllm.py:1089] Cudagraph is disabled under eager mode


In [8]:
ADAPTER_PATH = "./qlora_adapter"
qlora_model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"LoRA adapter saved to {ADAPTER_PATH}")


LoRA adapter saved to ./qlora_adapter


In [12]:
## 7. Generate Responses (problem-type routing)

Different question types reward different inference strategies, so we route prompts into three buckets and run each with its own `SamplingParams`:

| Type | N | temperature | rationale |
|---|---|---|---|
| **MCQ** | 1 | 0.2 | plug-and-check; self-consistency wastes budget when there are 4–10 fixed choices |
| **Free-form single** | 8 | 0.6 | heavy thinking; vote on the extracted `\boxed{}` value |
| **Free-form multi-blank** | 8 | 0.6 | every blank is an independent failure point — vote **per blank** at scoring time |

vLLM batches all prompts in a bucket together, so the three calls are still efficient.

SyntaxError: invalid character '–' (U+2013) (291824781.py, line 7)

In [9]:
# ── Problem-type routing ─────────────────────────────────────────────────────
# Different question types reward different inference strategies:
#   MCQ        → N=1, temperature=0.2   (plug-and-check; voting wastes budget)
#   FF-single  → N=8, temperature=0.6   (self-consistency on the boxed value)
#   FF-multi   → N=8, temperature=0.6   (per-blank voting at scoring time)
import time
from collections import Counter
from vllm import SamplingParams

LIMIT = 10                        # quick smoke test; set to None to run full 1126

data_run = data if LIMIT is None else data[:LIMIT]


def question_type(item):
    if item.get("options"):
        return "mcq"
    ans = item["answer"]
    if isinstance(ans, list) and len(ans) > 1:
        return "ff_multi"
    return "ff_single"


# Bucket prompts by type so each bucket can run with its own SamplingParams
buckets = {"mcq": [], "ff_single": [], "ff_multi": []}
for idx, item in enumerate(data_run):
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    buckets[question_type(item)].append((idx, prompt_text))

print(f"Buckets — MCQ: {len(buckets['mcq'])}, "
      f"FF-single: {len(buckets['ff_single'])}, "
      f"FF-multi: {len(buckets['ff_multi'])}")

sampling_mcq = SamplingParams(
    n=1,
    max_tokens=MAX_TOKENS,
    temperature=0.2,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
)

sampling_ff = SamplingParams(
    n=8,                          # self-consistency
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
)

samples_per_idx = [None] * len(data_run)   # list of N response strings per question


def run_bucket(name, params):
    bucket = buckets[name]
    if not bucket:
        return
    idxs, prompts = zip(*bucket)
    print(f"\nGenerating {name}: {len(prompts)} prompts × N={params.n}")
    t0 = time.perf_counter()
    outputs = llm.generate(list(prompts), sampling_params=params)
    print(f"  ⏱️ {time.perf_counter() - t0:.1f}s")
    for idx, out in zip(idxs, outputs):
        samples_per_idx[idx] = [c.text.strip() for c in out.outputs]


run_bucket("mcq",       sampling_mcq)
run_bucket("ff_single", sampling_ff)
run_bucket("ff_multi",  sampling_ff)

# Preview first sample of a few questions
for idx in {0, len(data_run) // 2, len(data_run) - 1}:
    qt = question_type(data_run[idx])
    n  = len(samples_per_idx[idx])
    print(f"\n── id={data_run[idx].get('id')} type={qt} N={n} ──")

Buckets — MCQ: 3, FF-single: 4, FF-multi: 3

Generating mcq: 3 prompts × N=1


Rendering prompts:   0%|          | 0/3 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/3 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  ⏱️ 253.0s

Generating ff_single: 4 prompts × N=8


Rendering prompts:   0%|          | 0/4 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/32 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  ⏱️ 180.1s

Generating ff_multi: 3 prompts × N=8


Rendering prompts:   0%|          | 0/3 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/24 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  ⏱️ 880.2s

── id=0 type=ff_single N=8 ──

── id=9 type=mcq N=1 ──

── id=5 type=ff_multi N=8 ──


In [ ]:
## 8. Score Responses

Scoring is self-consistency aware:

- **MCQ** — extract the predicted letter from each of N samples and majority-vote.
- **Free-form single** — extract `\boxed{...}` from each sample and majority-vote on the body (string match).
- **Free-form multi-blank** — split the boxed body by top-level commas and majority-vote **each blank slot independently**, then re-stitch into one `\boxed{a, b, c}` for the judger.

The voted answer is reassembled into a single `response` string, scored with `Judger.auto_judge()` for free-form. Each result record contains `{id, type, is_mcq, gold, response, samples, correct}`.

In [13]:


# ── Self-consistency aware scoring ────────────────────────────────────────────
# MCQ        → majority-vote on extracted letter
# FF-single  → majority-vote on extracted \boxed{} content (string match)
# FF-multi   → split top-level commas inside \boxed{} and vote each blank slot
#              independently — every blank is its own failure point.
import re
import sys
from collections import Counter
from tqdm import tqdm


def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def extract_boxed(text: str):
    """Return the body of the LAST \\boxed{...}, brace-balanced. None if absent."""
    idx = text.rfind("\\boxed{")
    if idx == -1:
        return None
    start = idx + len("\\boxed{")
    depth, i = 1, start
    while i < len(text):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                return text[start:i]
        i += 1
    return None


def split_blanks(boxed: str):
    """Split top-level commas; respect (), [], {} so '\\frac{1,2}{3}' stays one piece."""
    parts, cur, depth = [], [], 0
    for ch in (boxed or ""):
        if ch in "([{":
            depth += 1
        elif ch in ")]}":
            depth -= 1
        if ch == "," and depth == 0:
            parts.append("".join(cur).strip())
            cur = []
        else:
            cur.append(ch)
    if cur:
        parts.append("".join(cur).strip())
    return parts


def vote(items):
    """Plurality vote, ignoring empty/None. Returns None if nothing usable."""
    items = [x for x in items if x]
    if not items:
        return None
    return Counter(items).most_common(1)[0][0]


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, samples in tqdm(zip(data_run, samples_per_idx),
                          total=len(data_run), desc="Scoring"):
    qtype = question_type(item)
    gold  = item["answer"]
    samples = samples or []   # guard against None from a missing bucket

    if qtype == "mcq":
        voted     = vote([extract_letter(s) for s in samples]) or ""
        canonical = f"Therefore, the answer is \\boxed{{{voted}}}."
        correct   = voted == str(gold).strip().upper()

    elif qtype == "ff_single":
        voted     = vote([(extract_boxed(s) or "").strip() for s in samples]) or ""
        canonical = f"Therefore, the final answer is \\boxed{{{voted}}}."
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=canonical,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    else:  # ff_multi: per-blank voting beats voting on the whole tuple
        n_blanks   = len(gold) if isinstance(gold, list) else 1
        per_sample = [split_blanks(extract_boxed(s) or "") for s in samples]
        voted_blanks = []
        for k in range(n_blanks):
            col = [p[k] if k < len(p) else "" for p in per_sample]
            voted_blanks.append(vote(col) or "")
        canonical = f"Therefore, the final answer is \\boxed{{{', '.join(voted_blanks)}}}."
        try:
            correct = judger.auto_judge(
                pred=canonical,
                gold=gold,
                options=[[]] * n_blanks,
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "type":     qtype,
        "is_mcq":   qtype == "mcq",
        "gold":     gold,
        "response": canonical,
        "samples":  samples,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")


Scoring:   0%|          | 0/10 [00:00<?, ?it/s]

Scoring:  30%|███       | 3/10 [00:00<00:00, 11.38it/s]

Scoring:  60%|██████    | 6/10 [00:00<00:00, 17.59it/s]

Scoring:  90%|█████████ | 9/10 [00:00<00:00, 21.14it/s]

Scoring: 100%|██████████| 10/10 [00:00<00:00, 20.95it/s]

Scoring complete. 10 results.


## 9. Summary

Print accuracy broken down by question type (MCQ / FF single / FF multi).

In [14]:
mcq_res       = [r for r in results if r["type"] == "mcq"]
ff_single_res = [r for r in results if r["type"] == "ff_single"]
ff_multi_res  = [r for r in results if r["type"] == "ff_multi"]


def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0


print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  FF single  : {sum(r['correct'] for r in ff_single_res):4d} / {len(ff_single_res):4d}  ({acc(ff_single_res):.2f}%)")
print(f"  FF multi   : {sum(r['correct'] for r in ff_multi_res):4d} / {len(ff_multi_res):4d}  ({acc(ff_multi_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)


EVALUATION RESULTS
  MCQ        :    2 /    3  (66.67%)
  FF single  :    3 /    4  (75.00%)
  FF multi   :    2 /    3  (66.67%)
  Overall    :    7 /   10  (70.00%)


In [11]:
## 10. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, response, gold, correct}`.

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — toggle `SAVE_EVAL = False`.

Set `SAVE_SAMPLES = True` to also keep all N raw samples per question (useful for offline analysis but bigger files).

SyntaxError: invalid character '—' (U+2014) (67729406.py, line 5)

In [12]:
SAVE_EVAL    = True    # Set to False on the private test set (no ground truth)
SAVE_SAMPLES = False   # Set True to keep all N raw samples per question

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        if SAVE_EVAL:
            record["gold"]    = r["gold"]
            record["correct"] = r["correct"]
        if SAVE_SAMPLES:
            record["samples"] = r["samples"]
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")


NameError: name 'results' is not defined

SAVE_EVAL    = True    # Set to False on the private test set (no ground truth)
SAVE_SAMPLES = False   # Set True to keep all N raw samples per question

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        if SAVE_EVAL:
            record["gold"]    = r["gold"]
            record["correct"] = r["correct"]
        if SAVE_SAMPLES:
            record["samples"] = r["samples"]
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")


In [13]:
## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!

EVALUATION RESULTS
  MCQ        :    2 /    3  (66.67%)
  Free-form  :    5 /    7  (71.43%)
  Overall    :    7 /   10  (70.00%)
